# Task 2 - Motor de Inferencia
### José Antonio Mérida Castejón - 201105

## Task 2.0 - Modelado

Modelamos este problema de inferencia como una **Red Bayesiana** sobre tres variables aleatorias binarias: $B$ (robo), $E$ (terremoto) y $A$ (alarma). La estructura del grafo causal es:

$$B \rightarrow A \leftarrow E$$

### 2.0.1 Parámetros del modelo

| Variable | Significado | Probabilidad |
|----------|-------------|--------------|
| $B$ | Robo (*Burglary*) | Prior: $P(B=1) = \epsilon = 0.01$ |
| $E$ | Terremoto (*Earthquake*) | Prior: $P(E=1) = \epsilon = 0.01$ |
| $A$ | Alarma (*Alarm*) | CPT: $P(A=1 \mid B,E) = \mathbf{1}[B \vee E]$ |

En otras palabras, la alarma suena si y solo si ocurrió un robo o un terremoto.

### 2.0.2 Probabilidades de Nodos Raíz

Las variables $B$ y $E$ son **nodos raíz**, no tienen padres en el grafo. Entonces sus probabilidades se especifican directamente como priors, ya que no dependen de otras variables.

In [46]:
EPSILON = 0.01

def prob_b(b): return EPSILON if b == 1 else 1 - EPSILON
def prob_e(e): return EPSILON if e == 1 else 1 - EPSILON

### 2.0.3 Probabilidad del Nodo Hijo

$A$ es un **nodo hijo**, su probabilidad está condicionada en sus padres $B$ y $E$. Entonces se especifica mediante una CPT en lugar de priors directamente, aplicando la regla detallada en el laboratorio dónde la alarma suena si y solo si ocurrió un robo o un terremoto.

In [47]:
def prob_a_dado_b_e(a,b,e): return 1.0 if a == int(b or e) else 0.0

## Task 2.1 - Generador de Distribución Conjunta

Por la regla de la cadena para Redes Bayesianas:

$$P(B, E, A) = P(B) \cdot P(E) \cdot P(A \mid B, E)$$

In [48]:
def prob_conjunta(b,e,a): return prob_b(b) * prob_e(e) * prob_a_dado_b_e(a,b,e)

## Task 2.2 — Inferencia Marginal

En esencia, es simplemente la regla de Bayes:

$$P(\text{query} \mid \text{evidencia}) = \frac{P(\text{query},\ \text{evidencia})}{P(\text{evidencia})}$$

Enumeramos las $2^3 = 8$ combinaciones posibles de $(B, E, A)$, y por cada una que concuerde con la evidencia acumulamos la probabilidad conjunta en el denominador. Si además concuerda con la query, también va al numerador.

In [49]:
def inferencia_marginal(query: dict, evidencia: dict) -> float:

    numerador   = 0.0   # acumula P(query, evidencia)
    denominador = 0.0   # acumula P(evidencia)

    # Iterar por las 8 combinaciones
    for b in [0,1]:
        for e in [0,1]:
            for a in [0,1]:
                # Inicializar asignacion
                asignacion = {'B': b, 'E': e, 'A': a}

                # Descartar combinaciones que no concuerdan con la evidencia
                if not all(asignacion[v] == val for v, val in evidencia.items()):
                    continue

                # Obtener probabilidad conjunta y sumar a denominador (P(evidencia))
                p = prob_conjunta(b, e, a)
                denominador += p

                # Sumar a numerador si es consistente concuerda con el query (P(query, evidencia))
                if all(asignacion[v] == val for v, val in query.items()):
                    numerador += p

    return numerador / denominador



## Task 2.2.1 — Prior del Efecto

Calculamos $P(A=1)$ sumando los tres escenarios donde la alarma puede sonar:

$$P(A=1) = P(B=1)P(E=0) + P(B=0)P(E=1) + P(B=1)P(E=1)$$
$$= (0.01)(0.99) + (0.99)(0.01) + (0.01)(0.01) = 0.0199$$

Usando `inferencia_marginal` con evidencia vacía, la función itera sobre las 8 combinaciones de $(B, E, A)$, conserva solo las que tienen $A=1$, y suma sus probabilidades conjuntas.

In [50]:
p_alarma = inferencia_marginal(query={'A': 1}, evidencia={})

print(f"P(A=1) = {p_alarma:.4f}")

P(A=1) = 0.0199


Podemos ver que el resultado es consistente con el análisis teórico, por lo que podemos concluir que la implementación fue correcta.

## Task 2.3 — Efecto *Explain Away*

Cuando dos causas independientes $B$ y $E$ comparten un efecto común $A$, observar $A=1$ 
crea una dependencia implícita entre ellas. Si luego confirmamos $E=1$, el terremoto 
"explica" la alarma y reduce la necesidad de invocar el robo como causa.

Calculamos:

$$P(B=1 \mid A=1) \quad \text{vs} \quad P(B=1 \mid A=1, E=1)$$

### 2.3.1 Diagnóstico Simple

In [51]:
p_robo_dado_alarma = inferencia_marginal(
    query     = {'B': 1},
    evidencia = {'A': 1}
)

print(f"P(B=1 | A=1) = {p_robo_dado_alarma:.4f}")

P(B=1 | A=1) = 0.5025


La alarma sonó y no sabemos nada más, entonces tanto el robo como el terremoto son igualmente sospechosos. Por lo tanto, la probabilidad de robo sube de $\epsilon = 0.01$ a $\approx 0.50$ si sabemos que la alarma sonó.

### 2.3.2 Efecto Explain Away

In [52]:
p_robo_dado_alarma_y_terremoto = inferencia_marginal(
    query     = {'B': 1},
    evidencia = {'A': 1, 'E': 1}
)

print(f"P(B=1 | A=1, E=1) = {p_robo_dado_alarma_y_terremoto:.4f}")

P(B=1 | A=1, E=1) = 0.0100


Al confirmar que hubo un terremoto, este "explica" completamente la alarma y la probabilidad de robo colapsa de vuelta a su prior $\epsilon = 0.01$ como si no hubiéramos escuchado la alarma.

## Conclusión

### Teoría

$B$ y $E$ son **marginalmente independientes**. Sin evidencia, conocer uno no dice nada del otro. Pero al observar su efecto común $A=1$, se abre un camino de dependencia entre ellos. Confirmar $E=1$ "consume" la explicación de la alarma, reduciendo la necesidad de invocar $B$ como causa alternativa. Esto es el efecto **Explain Away**, especialmente observable en este caso donde tenemos completamente por seguro que fue el terremoto que causó la alarma. Por lo tanto, la probabilidad que ocurra $B$ simplemente es la probabilidad dado los priors ya que la evidencia ocurrió dada otro factor influyente.

Además, antes de entrar a la sección de cálculos podemos intuir como funciona este efecto con un simple argumento de combinatoria. Dado que la probabilidad de $A$ se enfoca en un OR lógico entre las demás variables, si sabemos que una ocurrió debemos remover el caso dónde $B$ ocurre pero no $E$.

### Cálculo

**Diagnóstico simple** $P(B=1 \mid A=1)$: trivialmente, solo 3 de las 8 combinaciones tienen $A=1$:

| $B$ | $E$ | $P(B,E,A=1)$ |
|-----|-----|--------------|
| 1   | 0   | $0.00990$ |
| 0   | 1   | $0.00990$ |
| 1   | 1   | $0.00010$ |

$$P(B=1 \mid A=1) = \frac{0.00990 + 0.00010}{0.01990} \approx 0.5025$$

**Explain Away** $P(B=1 \mid A=1, E=1)$: fijar $E=1$ deja solo 2 combinaciones válidas:

| $B$ | $E$ | $P(B,E=1,A=1)$ |
|-----|-----|----------------|
| 0   | 1   | $0.00990$ |
| 1   | 1   | $0.00010$ |

$$P(B=1 \mid A=1, E=1) = \frac{0.00010}{0.01000} = 0.0100$$

### Resultado

| Query | Resultado |
|-------|-----------|
| $P(B=1 \mid A=1)$ | `0.5025` |
| $P(B=1 \mid A=1, E=1)$ | `0.0100` |